In [2]:
import glob, numpy as np, pandas as pd, torch, timm
from sklearn.metrics import roc_auc_score

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
PLANES = ['sagittal', 'coronal', 'axial']
BASE_IN = '/kaggle/input/notebooks/alextanasescu/02-preprocess-test-alex'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# val knees for each plane (val labels in these manifests are the real ones)
val_by_plane = {}
for p in PLANES:
    m = pd.read_csv(f'{BASE_IN}/manifest_{p}.csv')
    m['path'] = f'{BASE_IN}/images_{p}/' + m['StudyInstanceUID'] + '.npy'
    val_by_plane[p] = m[m['split'] == 'val'].reset_index(drop=True)

def load(name):
    f = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(f) == 1, (name, f)
    m = timm.create_model('resnet18', pretrained=False, num_classes=12, in_chans=1)
    m.load_state_dict(torch.load(f[0], map_location=DEVICE))
    return m.to(DEVICE).eval()

@torch.no_grad()
def knee_probs(model):   # mean over slices 6-18, then mean over planes = your submission recipe
    rows = []
    for p in PLANES:
        for _, r in val_by_plane[p].iterrows():
            x = np.load(r['path'])[6:19].astype(np.float32) / 255.0
            x = torch.from_numpy((x - 0.45) / 0.225).unsqueeze(1).to(DEVICE)
            rows.append([r['StudyInstanceUID'], *torch.sigmoid(model(x)).mean(0).cpu().numpy()])
    return pd.DataFrame(rows, columns=['StudyInstanceUID'] + LABELS).groupby('StudyInstanceUID').mean()

p14 = knee_probs(load('resnet18_random_slice_15ep.pt'))
p17 = knee_probs(load('resnet18_newlabels_v1_15ep.pt'))

truth = (pd.concat(val_by_plane.values()).drop_duplicates('StudyInstanceUID')
         .set_index('StudyInstanceUID').loc[p14.index, LABELS] >= 0.5)
auc = lambda p: pd.Series({l: roc_auc_score(truth[l], p.loc[truth.index, l]) for l in LABELS})

table = pd.DataFrame({'14 old': auc(p14), '17 new': auc(p17), 'avg 14+17': auc((p14 + p17) / 2)})
print(table.round(3))
print('\nmacro:', table.mean().round(4).to_dict())

                  14 old  17 new  avg 14+17
ACL                0.755   0.768      0.761
MCL                0.687   0.649      0.685
Medial Meniscus    0.656   0.663      0.663
Lateral Meniscus   0.835   0.834      0.839
Medial OA          0.913   0.912      0.924
Lateral OA         0.766   0.787      0.768
PF OA              0.775   0.780      0.779
Effusion           0.943   0.919      0.958
Synovitis          0.740   0.740      0.746
Baker's            0.788   0.768      0.784
Contusion          0.816   0.808      0.821
Fracture           0.786   0.742      0.778

macro: {'14 old': 0.7883, '17 new': 0.7808, 'avg 14+17': 0.792}


In [3]:
def load34(name):
    f = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(f) == 1, (name, f)
    m = timm.create_model('resnet34', pretrained=False, num_classes=12, in_chans=1)
    m.load_state_dict(torch.load(f[0], map_location=DEVICE))
    return m.to(DEVICE).eval()

p15 = knee_probs(load34('resnet34_random_slice_15ep.pt'))

table['15 r34'] = auc(p15)
table['avg 14+15'] = auc((p14 + p15) / 2)
table['avg 14+15+17'] = auc((p14 + p15 + p17) / 3)
print(table.round(3))
print('\nmacro:', table.mean().round(4).to_dict())

                  14 old  17 new  avg 14+17  15 r34  avg 14+15  avg 14+15+17
ACL                0.755   0.768      0.761   0.755      0.765         0.756
MCL                0.687   0.649      0.685   0.701      0.698         0.692
Medial Meniscus    0.656   0.663      0.663   0.625      0.638         0.645
Lateral Meniscus   0.835   0.834      0.839   0.811      0.825         0.827
Medial OA          0.913   0.912      0.924   0.913      0.919         0.918
Lateral OA         0.766   0.787      0.768   0.783      0.781         0.776
PF OA              0.775   0.780      0.779   0.777      0.777         0.782
Effusion           0.943   0.919      0.958   0.948      0.943         0.958
Synovitis          0.740   0.740      0.746   0.728      0.740         0.744
Baker's            0.788   0.768      0.784   0.774      0.784         0.784
Contusion          0.816   0.808      0.821   0.812      0.821         0.821
Fracture           0.786   0.742      0.778   0.768      0.783         0.781

In [5]:
for f in glob.glob('/kaggle/input/**/*.pt', recursive=True): print(f)

/kaggle/input/notebooks/alextanasescu/14-random-slices-15ep/resnet18_random_slice_15ep.pt
/kaggle/input/notebooks/alextanasescu/15-resnet34/resnet34_random_slice_15ep.pt
/kaggle/input/notebooks/alextanasescu/17-new-labels/resnet18_newlabels_v1_15ep.pt
/kaggle/input/notebooks/alextanasescu/17v2-labels/resnet18_newlabels_v2_15ep.pt


In [6]:
p17v2 = knee_probs(load('resnet18_newlabels_v2_15ep.pt'))

table['17v2 new'] = auc(p17v2)
table['avg 14+15+17v2'] = auc((p14 + p15 + p17v2) / 3)
table['avg 14+15+17+17v2'] = auc((p14 + p15 + p17 + p17v2) / 4)
print(table[['14 old', '17 new', '17v2 new', 'avg 14+15+17', 'avg 14+15+17v2', 'avg 14+15+17+17v2']].round(3))
print('\nmacro:', table.mean().round(4).to_dict())

                  14 old  17 new  17v2 new  avg 14+15+17  avg 14+15+17v2  \
ACL                0.755   0.768     0.750         0.756           0.761   
MCL                0.687   0.649     0.605         0.692           0.683   
Medial Meniscus    0.656   0.663     0.679         0.645           0.647   
Lateral Meniscus   0.835   0.834     0.820         0.827           0.827   
Medial OA          0.913   0.912     0.922         0.918           0.921   
Lateral OA         0.766   0.787     0.779         0.776           0.772   
PF OA              0.775   0.780     0.761         0.782           0.773   
Effusion           0.943   0.919     0.939         0.958           0.943   
Synovitis          0.740   0.740     0.821         0.744           0.774   
Baker's            0.788   0.768     0.759         0.784           0.781   
Contusion          0.816   0.808     0.779         0.821           0.811   
Fracture           0.786   0.742     0.707         0.781           0.772   

           